# Storm Best-Track Map (single ATCF file)

**Render one ATCF best-track `.dat` file into a publication-style map.**

This notebook is the single-file companion to `plot_tracks.py`. Point it at **one** `.dat`
file and it draws the track colored by 1-minute maximum wind, with start/end markers and an
intensity legend.

It runs locally and on Google Colab. On Colab (or any machine without the file) you can set an
optional download link and the notebook fetches the `.dat` file first.

**Pipeline:**

```
one ATCF .dat file  ->  parse advisories  ->  render map  ->  outputs/track_plots/*_best_track.png
```


## 0. Environment

Google Colab does not ship `cartopy`, so install it there on first use. Locally the package is
assumed to be installed already.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    # cartopy is not preinstalled on Colab; install quietly if missing.
    try:
        import cartopy  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cartopy"], check=False)

print("IN_COLAB:", IN_COLAB)

## 1. Setup --- imports, constants, helpers

`INTENSITY_STYLES` holds the Saffir--Simpson bands (label, wind range in knots, color) used to
color the track. The helper functions parse ATCF records and prepare the map styling.

In [ ]:
%matplotlib inline

import csv
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ATCF timestamp format (YYYYMMDDHH).
ATCF_DATETIME_FORMAT = "%Y%m%d%H"

# Saffir--Simpson intensity bands: (label, lower kt, upper kt, color).
INTENSITY_STYLES = (
    ("Tropical depression", 0, 33, "#6b7280"),
    ("Tropical storm", 34, 63, "#2b8cbe"),
    ("Category 1", 64, 82, "#e6ab02"),
    ("Category 2", 83, 95, "#e67e22"),
    ("Category 3", 96, 112, "#d95f0e"),
    ("Category 4", 113, 136, "#c62828"),
    ("Category 5", 137, float("inf"), "#7b3294"),
)


def parse_coordinate(value: str) -> float:
    """Convert an ATCF tenths-of-a-degree coordinate such as 286N or 969W."""
    match = re.fullmatch(r"(\d+)([NSEW])", value.strip().upper())
    if not match:
        raise ValueError(f"Invalid ATCF coordinate: {value!r}")
    magnitude = int(match.group(1)) / 10.0
    return -magnitude if match.group(2) in {"S", "W"} else magnitude


def load_atcf_track(dat_path: Path) -> pd.DataFrame:
    """Load the core fields from an ATCF best-track file."""
    parsed_rows = []
    with dat_path.open(newline="", encoding="utf-8") as handle:
        for line_number, fields in enumerate(csv.reader(handle), start=1):
            fields = [field.strip() for field in fields]
            if not fields or all(not field for field in fields):
                continue
            if len(fields) < 11:
                raise ValueError(
                    f"{dat_path}:{line_number} has {len(fields)} fields; expected at least 11"
                )
            try:
                parsed_rows.append(
                    {
                        "basin": fields[0],
                        "storm_number": fields[1],
                        "time": pd.to_datetime(fields[2], format=ATCF_DATETIME_FORMAT, errors="raise"),
                        "technique": fields[4],
                        "latitude": parse_coordinate(fields[6]),
                        "longitude": parse_coordinate(fields[7]),
                        "vmax_kt": pd.to_numeric(fields[8], errors="coerce"),
                        "pressure_hpa": pd.to_numeric(fields[9], errors="coerce"),
                        "status": fields[10],
                    }
                )
            except ValueError as exc:
                raise ValueError(f"{dat_path}:{line_number}: {exc}") from exc

    track = pd.DataFrame(parsed_rows)
    if track.empty:
        raise ValueError(f"No track records found in {dat_path}")

    # ATCF files may repeat a timestamp for different wind-radii records; those
    # share a center position, so one point per advisory is enough here.
    track = (
        track.sort_values("time")
        .drop_duplicates(subset=["time"], keep="first")
        .reset_index(drop=True)
    )
    return track


def padded_extent(track: pd.DataFrame) -> list:
    lon_min = float(track["longitude"].min())
    lon_max = float(track["longitude"].max())
    lat_min = float(track["latitude"].min())
    lat_max = float(track["latitude"].max())
    lon_pad = max((lon_max - lon_min) * 0.12, 2.5)
    lat_pad = max((lat_max - lat_min) * 0.12, 2.0)
    return [lon_min - lon_pad, lon_max + lon_pad, lat_min - lat_pad, lat_max + lat_pad]


def intensity_label(vmax_kt) -> str:
    if pd.isna(vmax_kt):
        return "Unknown"
    for label, lower, upper, _color in INTENSITY_STYLES:
        if lower <= vmax_kt <= upper:
            return label
    return "Unknown"


def intensity_color(label: str) -> str:
    for candidate, _lower, _upper, color in INTENSITY_STYLES:
        if label == candidate:
            return color
    return "#9ca3af"


def add_base_features(ax) -> None:
    """Apply the same Natural Earth visual language as the station-map notebook."""
    ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="#D8DBDE", zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#FFFFFF", zorder=0)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), facecolor="#D8DBDE",
                   edgecolor="#A8A8A8", linewidth=0.25, zorder=1)
    ax.add_feature(cfeature.RIVERS.with_scale("10m"), edgecolor="#A8A8A8",
                   linewidth=0.22, alpha=0.75, zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), edgecolor="#A8A8A8",
                   linewidth=0.45, alpha=0.65, zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor="#CFCFCF",
                   linewidth=0.30, alpha=0.55, zorder=2)
    ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="#CFCFCF",
                   linewidth=0.30, alpha=0.55, zorder=2)


def clean_path_part(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())

## 2. Paths

`OUTPUT_DIR` is where the map is written. It is chosen automatically: under `/content` on
Google Colab, and next to the notebook locally (override with the `HYDRO_WORKDIR` environment
variable).

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Project root (holds code/ and data/). Detected via the always-present code/
# directory, so it is correct even before any data has been downloaded.
_cwd = Path.cwd().resolve()
if "HYDRO_WORKDIR" in os.environ:
    WORKDIR = Path(os.environ["HYDRO_WORKDIR"]).resolve()
elif IN_COLAB:
    WORKDIR = Path("/content/per_storm")
elif (_cwd.parent / "code").is_dir():
    WORKDIR = _cwd.parent          # run from .../code/
elif (_cwd / "code").is_dir():
    WORKDIR = _cwd                 # run from the project root
else:
    WORKDIR = _cwd

DATA_ROOT  = WORKDIR / "data"
OUTPUT_DIR = WORKDIR / "outputs" / "track_plots"

print("IN_COLAB  :", IN_COLAB)
print("WORKDIR   :", WORKDIR)
print("DATA_ROOT :", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 3. Input --- the one `.dat` file

Edit the cell below:

- **`DAT_PATH`** --- the ATCF best-track file to plot.
- **`DAT_URL`** --- *optional.* A download link (e.g. a Nextcloud public share) used only if
  `DAT_PATH` is missing. Leave `""` to always use the local file.
- **`DAT_SHA256`** --- *optional.* SHA-256 of the downloaded file; `""` disables the check.
- **`STORM_NAME` / `YEAR`** --- *optional.* Used only for the map title and output filename.
  ATCF files store the basin code and storm **number**, not the name, so these default to a
  generic title.

In [ ]:
# ----------------------------------------------------------------------
# INPUT
# ----------------------------------------------------------------------
DAT_PATH = DATA_ROOT / "tracks" / "2024_MILTON" / "bal142024.dat"

STORM_NAME = "Milton"   # optional display name ("" -> generic title)
YEAR       = "2024"      # optional year ("" -> omitted from the title)

DPI           = 300      # raster output DPI
OUTPUT_FORMAT = "png"    # png / pdf / svg / tiff

print("DAT_PATH:", DAT_PATH)


## 4. Data acquisition

If the file is already on disk it is used as-is. Otherwise, if `DAT_URL` is set, it is
downloaded to `DAT_PATH` (with the same SSL workaround the hydrograph notebook uses for the
CERA host, plus an optional SHA-256 check).

In [ ]:
# ----------------------------------------------------------------------
# Data acquisition: fetch the single self-contained archive
# (hydrographs + tracks) if it is not already on disk.
#
#   DATA_ROOT/hydrographs/{YEAR}/{STORM}/{MESH}/CERA_Hydrographs_.../hydrographs/*.json
#   DATA_ROOT/tracks/{YEAR}_{STORM}/bal*.dat
#
# Runs on Google Colab and locally. If the data is already present it is skipped.
# ----------------------------------------------------------------------
import hashlib
import ssl
import urllib.request
import zipfile

# Single public download link for per_storm_data.zip (fill in for Colab).
DATA_URL    = "https://cloud.cera.lsu.edu/public.php/dav/files/cq8Tb34b63xjycY/?accept=zip"
# SHA-256 of that archive, verified after download to guard integrity.
DATA_SHA256 = "a090b33f56bab601674e3df092e8cea65604ce9656518b0ae7731e9ed8393657"


def _download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    # The CERA host currently serves an expired TLS certificate, so verification
    # is disabled here; the SHA-256 check below is what protects integrity.
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(url, context=ctx) as resp, open(dest, "wb") as fh:
        fh.write(resp.read())


have_data = (DATA_ROOT / "hydrographs").is_dir() and (DATA_ROOT / "tracks").is_dir()
if have_data:
    print("Data already present:", DATA_ROOT)
elif DATA_URL:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / "per_storm_data.zip"
    print("Downloading", DATA_URL, "->", zip_path)
    _download(DATA_URL, zip_path)
    if DATA_SHA256:
        digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
        assert digest == DATA_SHA256, f"checksum mismatch:\n got {digest}\n exp {DATA_SHA256}"
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)   # archive top level: hydrographs/ + tracks/
    print("Extracted to", DATA_ROOT)
else:
    raise FileNotFoundError(
        f"Data missing under {DATA_ROOT} and DATA_URL is empty. "
        f"Set DATA_URL to the per_storm_data.zip link."
    )

# Sanity check.
print("Packages found:", len(list((DATA_ROOT / "hydrographs").glob("*/*/*/CERA_Hydrographs_*"))))
print("Track DATs found:", len(list((DATA_ROOT / "tracks").glob("*_*/bal*.dat"))))


## 5. Render the map

Parses the advisories, colors each point by intensity, draws the connecting track line, and
marks the start (green circle) and end (black X). The `Intensity (1-min wind)` legend lists
only the bands that actually occur in the file.

In [ ]:
def plot_track_map(
    dat_path: Path,
    output_dir: Path,
    storm_name: str = "",
    year: str = "",
    dpi: int = 300,
    output_format: str = "png",
):
    """Render a single ATCF best-track file into a publication-style map."""
    track = load_atcf_track(dat_path)
    track["intensity"] = track["vmax_kt"].map(intensity_label)

    plt.rcParams.update({
        "figure.dpi": 130,
        "savefig.dpi": 600,
        "font.family": "DejaVu Sans",
        "axes.titleweight": "bold",
        "axes.labelsize": 10,
        "axes.titlesize": 14,
        "legend.fontsize": 8,
    })

    projection = ccrs.PlateCarree()
    fig = plt.figure(figsize=(9.2, 6.8), constrained_layout=False)
    ax = fig.add_axes([0.05, 0.085, 0.72, 0.78], projection=projection)
    ax.set_extent(padded_extent(track), crs=projection)
    add_base_features(ax)

    gridlines = ax.gridlines(draw_labels=False, linewidth=0.28, color="#9aa5ad",
                             alpha=0.42, linestyle=(0, (2, 3)))

    for spine in ax.spines.values():
        spine.set_linewidth(0.55)
        spine.set_edgecolor("#4b5563")

    line = ax.plot(
        track["longitude"], track["latitude"], transform=projection,
        color="#334155", linewidth=1.65, alpha=0.92, zorder=5,
    )[0]
    line.set_path_effects([pe.Stroke(linewidth=3.2, foreground="white", alpha=0.8), pe.Normal()])

    present_labels = []
    for label, _lower, _upper, color in INTENSITY_STYLES:
        group = track[track["intensity"] == label]
        if group.empty:
            continue
        present_labels.append(label)
        ax.scatter(
            group["longitude"], group["latitude"], transform=projection, s=34, marker="o",
            facecolor=color, edgecolor="white", linewidth=0.6, alpha=0.96, zorder=6,
            path_effects=[pe.withStroke(linewidth=1.1, foreground="#263238", alpha=0.28)],
        )

    start = track.iloc[0]
    end = track.iloc[-1]
    ax.scatter([start["longitude"]], [start["latitude"]], transform=projection, s=72,
               marker="o", facecolor="none", edgecolor="#166534", linewidth=1.5, zorder=7)
    ax.scatter([end["longitude"]], [end["latitude"]], transform=projection, s=58,
               marker="X", facecolor="#111827", edgecolor="white", linewidth=0.55, zorder=7)

    max_wind = track["vmax_kt"].max()
    min_pressure = track.loc[track["pressure_hpa"] > 0, "pressure_hpa"].min()
    max_wind_text = "n/a" if pd.isna(max_wind) else f"{int(max_wind)} kt"
    pressure_text = "n/a" if pd.isna(min_pressure) else f"{int(min_pressure)} hPa"

    title = f"{storm_name.title()} ({year}) Best Track" if storm_name else "ATC Best Track"
    fig.text(0.50, 0.942, title, ha="center", va="center", fontsize=18,
             fontweight="bold", color="#111827")
    fig.text(0.50, 0.906,
             f"Advisories: {len(track)} \u00b7 Max wind: {max_wind_text} \u00b7 Min pressure: {pressure_text}",
             ha="center", va="center", fontsize=11.5, color="#4b5563")

    intensity_handles = [
        mlines.Line2D([], [], marker="o", linestyle="None", markersize=6.2,
                      markerfacecolor=intensity_color(label), markeredgecolor="white",
                      markeredgewidth=0.55, label=label)
        for label in present_labels
    ]
    endpoint_handles = [
        mlines.Line2D([], [], marker="o", linestyle="None", markersize=7,
                      markerfacecolor="none", markeredgecolor="#166534",
                      markeredgewidth=1.25, label="Track start"),
        mlines.Line2D([], [], marker="X", linestyle="None", markersize=6.5,
                      markerfacecolor="#111827", markeredgecolor="white",
                      markeredgewidth=0.5, label="Track end"),
    ]
    ax.legend(
        handles=[*intensity_handles, *endpoint_handles], title="Intensity (1-min wind)",
        loc="center left", bbox_to_anchor=(1.02, 0.2), frameon=True, framealpha=0.95,
        facecolor="white", edgecolor="#d0d5da", borderpad=0.5, labelspacing=0.42,
        handletextpad=0.55, fontsize=8.6, title_fontsize=9.0,
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    named = [p for p in (year, storm_name) if p] if storm_name else [dat_path.stem]
    stem = "_".join(clean_path_part(part) for part in [*named, "best_track"])
    output_path = output_dir / f"{stem}.{output_format}"
    fig.savefig(output_path, dpi=dpi, facecolor="white")
    plt.close(fig)
    print("Saved:", output_path)
    return output_path

## 6. Render and display

The map is exported to `OUTPUT_DIR` and then shown inline. We display the saved PNG directly
(a `cartopy` GeoAxes does not render reliably through the notebook's inline `plt.show()` path,
which can collapse the figure to just its title).

In [ ]:
from IPython.display import Image, display

output_path = plot_track_map(
    dat_path=DAT_PATH,
    output_dir=OUTPUT_DIR,
    storm_name=STORM_NAME,
    year=YEAR,
    dpi=DPI,
    output_format=OUTPUT_FORMAT,
)
display(Image(filename=str(output_path)))

## 6. Done

The map is written to `OUTPUT_DIR`. To plot a different storm, change `DAT_PATH` (and optionally
`STORM_NAME` / `YEAR`) in the **Input** cell and re-run from there.

> ATCF best-track files are provided by NHC (HURDAT2-derived); the same `.dat` files used by
> `scripts/track_plotter/plot_tracks.py` live under `data/tracks/{YEAR}_{STORM}/`.
